# Contrastive Representation Distillation: CIFAR-100 reproduction

This notebook reproduces the resnet32x4 → resnet8x4 row of Table 1 of Tian, Krishnan & Isola, [*Contrastive Representation Distillation*](https://arxiv.org/abs/1910.10699) (ICLR 2020), with the authors' code ([RepDistiller](https://github.com/HobbitLong/RepDistiller)):

| Model | Paper test top-1 (%, last epoch) |
|---|---|
| Teacher: resnet32x4 | 79.42 |
| Student: resnet8x4, cross-entropy only (vanilla) | 72.50 |
| Student: resnet8x4, KD (Hinton et al., 2015) | 73.33 |
| Student: resnet8x4, CRD | 75.51 |

Running every cell from top to bottom trains the teacher, the vanilla student, the KD student and the CRD student, then shows a results table and the training curves. The other methods of Table 1 (FitNet, AT, SP, CC, VID, RKD, PKT, AB, FT, FSP, NST) can be added to `STAGES` below. Every run also logs to Weights & Biases (project `distillation-crd`).

**How to open it.** On the GPU server, from the repository root (the uv environment is the one shared with `mnist/`):

```bash
uv sync
uv run wandb login
uv run --with jupyter jupyter lab CRD_ICLR2020/reproduce.ipynb
```

One seed takes several hours (teacher + 3 students, 240 epochs each), so start Jupyter inside `tmux`, or run the same stages with `uv run CRD_ICLR2020/scripts/run_experiment.py` in a terminal (see [SERVER.md](SERVER.md)): the notebook then reuses the finished runs instead of training again.

The training code is `train_teacher.py`, `train_student.py` and `scripts/run_experiment.py`; the notebook only calls `run_stage` from the latter, so it trains exactly what the command-line runs train.

## Setup

In [ ]:
import json
import statistics
import sys
import urllib.request
from pathlib import Path

# Works whether Jupyter was started in the repository root or in CRD_ICLR2020/.
HERE = next(p for p in (Path.cwd(), Path.cwd() / "CRD_ICLR2020") if (p / "train_student.py").exists())
sys.path.insert(0, str(HERE / "scripts"))

import matplotlib.pyplot as plt
import torch
from IPython.display import Markdown, display

import check_environment
from run_experiment import checkpoint, read_metrics, run_name, run_stage, sha256, stage_settings

print("torch", torch.__version__, "| CUDA available:", torch.cuda.is_available())

In [ ]:
SMOKE = False          # True: every run is 1 epoch of 2 batches, only to check that the pipeline works
SEEDS = [0]            # the paper reports one run per method; use [0, 1, 2] for a mean and std
TRIAL = 1              # a new trial number starts every run again in new folders
STAGES = ["vanilla", "kd", "crd"]   # more: "crd_kd", "fitnet", "at", "sp", "cc", "vid", "rkd", "pkt", "ab", "ft", "fsp", "nst"
TEACHER_SOURCE = "train"            # "train": train resnet32x4 here; "upstream": download the authors' teacher
REUSE = True           # reuse a finished run whose config and teacher match (re-run the plots without retraining)
QUIET = True           # print one summary per epoch; runs/<run>/console.log always has the full output

WANDB = True           # needs `wandb login` first; WANDB_MODE=offline also works
WANDB_PROJECT = "distillation-crd"
WANDB_ENTITY = ""      # W&B user or team; "" = your own account

CONFIG = HERE / "configs" / "cifar100.json"

In [ ]:
# Fails here, not hours later, if the GPU or the CUDA wheel does not work.
check_environment.main()

## Settings

The settings are those of the authors' scripts (`scripts/run_cifar_distill.sh` and the defaults of `train_student.py`), stored in `configs/cifar100.json` and shown below. Edit that file (or a copy, with `CONFIG` pointing to it) to change them.

| Setting | Value |
|---|---|
| Dataset | CIFAR-100: 50,000 training and 10,000 test images, 100 classes; random crop + horizontal flip |
| Optimizer | SGD, momentum 0.9, weight decay 5·10⁻⁴, batch 64 |
| Learning rate | 0.05, × 0.1 after epochs 150, 180 and 210; 240 epochs |
| Student loss | $\mathcal{L} = \gamma\,\mathrm{CE}(y, p_s) + \alpha\,T^2\,\mathrm{KL}(p_t^{(T)} \,\|\, p_s^{(T)}) + \beta\,\mathcal{L}_{\text{CRD}}$ |
| KD | γ = 0.1, α = 0.9, β = 0, T = 4 |
| CRD | γ = 1, α = 0, β = 0.8 |
| CRD+KD | γ = 1, α = 1, β = 0.8 |
| CRD critic | linear embeddings to 128 dimensions, L2-normalized; NCE temperature 0.07 |
| CRD negatives | K = 16,384 per sample, from a memory bank of all 50,000 training images updated with momentum 0.5 |
| Other methods | γ = 1, α = 0, β: FitNet 100 (hint on stage 2), AT 1000, SP 3000, CC 0.02, VID 1, RKD 1, PKT 30000, AB 1, FT 200, FSP 50, NST 50 |
| AB, FT, FSP | first 30 epochs of pretraining of the connector / paraphraser / student (`init_epochs`), then distillation |

$\mathcal{L}_{\text{CRD}}$ is the paper's NCE loss on the last-layer features: for each image, the critic should tell the teacher feature of the same image (positive) from the teacher features of K images of other classes (negatives, `mode = exact`, taken from the memory bank), and the same with teacher and student swapped; the two terms are added.

The teacher is evaluated before each student is trained; its accuracy appears in the output and in W&B as `teacher_acc`.

In [ ]:
config = json.loads(CONFIG.read_text(encoding="utf-8"))
display(Markdown("\n".join(["| Key | Value |", "|---|---|"] + [f"| `{k}` | {v} |" for k, v in config.items()])))

In [ ]:
def run(stage, seed, teacher=None):
    # Train one stage, or reuse its saved run when REUSE is on and its config and teacher match.
    folder = HERE / "runs" / run_name(stage, seed, TRIAL, SMOKE)
    manifest_file = folder / "manifest.json"
    if folder.exists():
        manifest = json.loads(manifest_file.read_text(encoding="utf-8")) if manifest_file.exists() else {}
        finished = manifest.get("status") in ("completed", "smoke_passed")
        same = (stage_settings(manifest.get("config", {}), stage) == stage_settings(config, stage)
                and manifest.get("teacher_sha256") == (sha256(teacher) if teacher else None))
        if REUSE and finished and same:
            print(f"{folder.name}: reusing (last test top-1 {manifest['last_epoch']['test_acc']:.2f})")
            return folder
        raise RuntimeError(f"{folder} exists (status {manifest.get('status')!r}, same settings: {finished and same}); "
                           "set a new TRIAL, or delete the folder yourself if it is a failed run")
    run_stage(stage, CONFIG, teacher, seed=seed, trial=TRIAL, smoke=SMOKE, wandb=WANDB,
              wandb_project=WANDB_PROJECT, wandb_entity=WANDB_ENTITY, echo=not QUIET)
    return folder


runs = {}   # (stage, seed) -> run folder

## 1. Teacher

`TEACHER_SOURCE = "train"` trains resnet32x4 with cross-entropy for 240 epochs (paper: 79.42%). `"upstream"` downloads the checkpoint the authors released (`scripts/fetch_pretrained_teachers.sh`) instead, which is what the paper's students were distilled from. Every student below uses the same teacher; its SHA256 is stored in each run's `manifest.json`.

In [ ]:
if SMOKE or TEACHER_SOURCE == "train":
    runs["teacher", SEEDS[0]] = run("teacher", SEEDS[0])
    TEACHER = checkpoint(runs["teacher", SEEDS[0]])
else:
    TEACHER = HERE / "runs" / "upstream_teacher" / "resnet32x4_vanilla" / "ckpt_epoch_240.pth"
    if not TEACHER.exists():
        TEACHER.parent.mkdir(parents=True, exist_ok=True)
        urllib.request.urlretrieve("http://shape2prog.csail.mit.edu/repo/resnet32x4_vanilla/ckpt_epoch_240.pth", TEACHER)
print("teacher:", TEACHER)
print("SHA256: ", sha256(TEACHER))

## 2. Students

All students are resnet8x4 with the same optimizer and schedule; only the loss differs (see Settings). The vanilla student does not use the teacher.

In [ ]:
for seed in SEEDS:
    for stage in STAGES:
        print(f"===== {stage}, seed {seed} =====")
        runs[stage, seed] = run(stage, seed, None if stage == "vanilla" else TEACHER)

## 3. Results

In [ ]:
# Table 1 of the paper (resnet32x4 -> resnet8x4); CRD+KD is not reported for this pair.
PAPER = {"teacher": 79.42, "vanilla": 72.50, "kd": 73.33, "crd": 75.51, "crd_kd": None,
         "fitnet": 73.50, "at": 73.44, "sp": 72.94, "cc": 72.97, "vid": 73.09, "rkd": 71.90,
         "pkt": 73.64, "ab": 73.17, "ft": 72.86, "fsp": 72.62, "nst": 73.30}
history = {key: read_metrics(folder) for key, folder in runs.items()}

def mean_std(values):
    return f"{statistics.mean(values):.2f}" + (f" ± {statistics.stdev(values):.2f}" if len(values) > 1 else "")

rows = ["| Model | Test top-1, last epoch | Best epoch | Paper | Runs |", "|---|---|---|---|---|"]
for stage in ["teacher"] + STAGES:
    keys = [key for key in history if key[0] == stage]
    if keys:
        last = [history[key][-1]["test_acc"] for key in keys]
        best = [max(row["test_acc"] for row in history[key]) for key in keys]
        paper = PAPER[stage] if PAPER[stage] is not None else "—"
        rows.append(f"| {stage} | {mean_std(last)} | {mean_std(best)} | {paper} | {len(keys)} |")
display(Markdown("\n".join(rows)))

The paper reports the test accuracy after the last epoch, so compare the "last epoch" column with it; the best epoch is chosen on the test set and is shown only for reference.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
for (stage, seed), rows in history.items():
    epochs = [row["epoch"] for row in rows]
    axes[0].plot(epochs, [row["test_acc"] for row in rows], label=f"{stage} (seed {seed})")
    axes[1].plot(epochs, [row["train_acc"] for row in rows], label=f"{stage} (seed {seed})")
axes[0].set(xlabel="epoch", ylabel="test top-1 (%)", title="Test accuracy")
axes[1].set(xlabel="epoch", ylabel="train top-1 (%)", title="Training accuracy (augmented, train mode)")
for ax in axes:
    ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
# The unweighted loss terms of the distilled students: CE with the labels, KL to the teacher at T = 4,
# and the method's own distillation loss (CRD, FitNet, ...; 0 for KD).
distilled = [key for key in history if key[0] not in ("teacher", "vanilla")]
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for key in distilled:
    rows = history[key]
    for ax, term in zip(axes, ("loss_cls", "loss_div", "loss_kd")):
        ax.plot([row["epoch"] for row in rows], [row[term] for row in rows], label=f"{key[0]} (seed {key[1]})")
for ax, title in zip(axes, ("Cross-entropy with labels", "KL(teacher ‖ student) at T, ×T²", "Distillation loss (method's own)")):
    ax.set(xlabel="epoch", title=title)
    ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

## Notes

- **Where everything is.** Each run has its own folder `runs/<stage>_trial<N>_seed<S>/` with `manifest.json` (command, config, teacher SHA256, status, last and best epoch), `console.log`, `pip-freeze.txt`, and under `save/` the per-epoch `metrics.jsonl`, the weights and the W&B files. The same metrics are in W&B, one group per stage.
- **Run-to-run noise.** CIFAR-100 results change by a few tenths of a percent from seed to seed; compare differences larger than that, and use `SEEDS = [0, 1, 2]` for a mean and std.
- **Teacher.** A teacher trained here can differ from the authors' by a few tenths of a percent, and the students depend on it; `TEACHER_SOURCE = "upstream"` removes that difference.
- **No validation set.** As in the paper, every model is trained on all 50,000 training images and evaluated on the 10,000 test images.